## Importing Libraries

In [8]:
import glob
import json
from tqdm import tqdm
import random
import os
from groq import Groq

## Setup Files

In [ ]:
GROQ_KEY = os.getenv("GROQ_API_KEY")
PATIENT_PROFILES = glob.glob("./patient_profiles/patient_*.json")

PERFECT_GEN_PROMPT = "./prompts/perfect-diary_gen-prompt.txt"
PERFECT_SYS_PROMPT = "./prompts/perfect-diary_sys-prompt.txt"
INCONSISTANCE_GEN_PROMPT = "./prompts/inconsistancies-int_gen-prompt.txt"
INCONSISTANCE_SYS_PROMPT = "./prompts/inconsistancies-int_sys-prompt.txt"

ANALYSIS_PROMPT = "./prompts/analysis-prompt.txt"
ANALYSIS_SYS_PROMPT = "./prompts/analysis_sys-prompt.txt"

DIARY_TEMPLATE = "./diaries_template/diary_template.txt"
DIARY_EXAMPLES = "./diaries_template/diaries_ex.txt"

LAB_EXAMPLE = "./lab_examples/lab_examples.txt"
LAB_SCHEMA = "./lab_examples/lab_schema.txt"

OUTPUT_DIR = "./outputs/"
OUTPUT_EXP_DIR = "./outputs/diary-gen_experiment"
PERFECT_OUTPUT_FILE = "perfect-diary_patient"
ANALYSIS_OUTPUT_FILE = "analysis_patient"
INCONSISTANCE_OUTPUT_FILE = "inconsistancy-diary_patient"
TRACK_FILE = "parameter_patient"

MODEL = "groq/compound" # llama-3.3-70b-versatile, openai/gpt-oss-120b, groq/compound

CLIENT = Groq(api_key=GROQ_KEY)
TEMP = 0.7

## Setup Environment

In [3]:
## Setting evironment
os.makedirs(OUTPUT_DIR,exist_ok=True)

count = 0

for path in os.listdir(OUTPUT_DIR):
    if os.path.isdir(os.path.join(OUTPUT_DIR, path)):
        count += 1
        
os.makedirs(f"{OUTPUT_EXP_DIR}_{count}", exist_ok=True)

## Generating Diaries

In [11]:
count = 19

style_modes = [
    "narrative-dominant",
    "telegraphic-hospital-style",
    "exam-and-imaging-focused",
    "toxicity-focused",
    "psychosocial-emphasis"
]

length_modes = [
    "short",
    "medium",
    "long"
]

pbar = tqdm(total=len(PATIENT_PROFILES), desc="Generating synthetic clinical diaries")

for patient in PATIENT_PROFILES:
    print("Processing patient:", patient)
    with open(patient, "r", encoding="utf-8") as f, \
         open(PERFECT_GEN_PROMPT, "r", encoding="utf-8") as perfect_gen_prompt_file, \
         open(PERFECT_SYS_PROMPT, "r", encoding="utf-8") as perfect_sys_prompt_file, \
         open(INCONSISTANCE_GEN_PROMPT, "r", encoding="utf-8") as incons_gen_prompt_file, \
         open(INCONSISTANCE_SYS_PROMPT, "r", encoding="utf-8") as incons_sys_prompt_file, \
         open(DIARY_TEMPLATE, "r", encoding="utf-8") as diary_template_file, \
         open(DIARY_EXAMPLES, "r", encoding="utf-8") as diary_ex_file:
             
        patient_id = int(patient.split('_')[2].split('.')[0])
        
        if patient_id > 13:
            
            print(f"Generating diary for patient {patient_id}")
            
            selected_style = random.choice(style_modes)
            selected_length = random.choice(length_modes)
            
            print(f"Selected stylistic mode for patient {patient_id}: {selected_style}")
            print(f"Selected length mode for patient {patient_id}: {selected_length}")
            
            patient_data = json.load(f)
            base_perfect_gen_prompt = perfect_gen_prompt_file.read()
            perfect_sys_prompt = perfect_sys_prompt_file.read()
            diary_template = diary_template_file.read()
            diary_examples = diary_ex_file.read()
            
            perfect_prompt_w_template = base_perfect_gen_prompt.replace("{{TEMPLATE_TEXT}}", diary_template)
            perfect_prompt_w_patient = perfect_prompt_w_template.replace("{{PATIENT_PROF}}", json.dumps(patient_data))
            perfect_prompt_w_style = perfect_prompt_w_patient.replace("{{STYLISTIC_MODE}}", selected_style)
            perfect_prompt_w_length = perfect_prompt_w_style.replace("{{LENGTH_MODE}}", selected_length)
            perfect_prompt_final = perfect_prompt_w_length.replace("{{DIARIES_TEXT}}", diary_examples)
            
            completion = CLIENT.chat.completions.create(
                model=MODEL, # llama-3.3-70b-versatile, openai/gpt-oss-120b the prompt isn-t optimized for gpt-oss-120b
                messages=[
                    {
                        "role": "system",
                        "content": perfect_sys_prompt
                    },
                    {
                        "role": "user",
                        "content": perfect_prompt_final
                    }
                ],
                temperature=TEMP
            )
            perfect_result = completion.choices[0].message.content

            
            with open(f"{OUTPUT_EXP_DIR}_{count}/{PERFECT_OUTPUT_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
                o.write(f"{perfect_result}\n\n")
                print(f"Saved LLM output on {OUTPUT_EXP_DIR}_{count}/{PERFECT_OUTPUT_FILE}_{patient_id}.txt")
                
            base_inconsistency_gen_prompt = incons_gen_prompt_file.read()
            inconsistency_sys_prompt = incons_sys_prompt_file.read()
            
            inconsistency_prompt_final = base_inconsistency_gen_prompt.replace("{{CLEAN_DIARY}}", perfect_result)
            
            completion = CLIENT.chat.completions.create(
                model=MODEL, # llama-3.3-70b-versatile, openai/gpt-oss-120b the prompt isn-t optimized for gpt-oss-120b
                messages=[
                    {
                        "role": "system",
                        "content": inconsistency_sys_prompt
                    },
                    {
                        "role": "user",
                        "content": inconsistency_prompt_final
                    }
                ],
                temperature=TEMP
            )
            inconsistency_result = completion.choices[0].message.content
            
            with open(f"{OUTPUT_EXP_DIR}_{count}/{INCONSISTANCE_OUTPUT_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
                o.write(f"{inconsistency_result}\n\n")
                print(f"Saved LLM output on {OUTPUT_EXP_DIR}_{count}/{INCONSISTANCE_OUTPUT_FILE}_{patient_id}.txt")
            
            with open(f"{OUTPUT_EXP_DIR}_{count}/{TRACK_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
                o.write(f"Model: {MODEL}\n"
                        f"  Stylistic Mode: {selected_style}\n"
                        f"  Length Mode: {selected_length}\n"
                        f"  Temperature: {TEMP}\n"
                        f"\n"
                        f"Perfect diary system prompt:\n{perfect_sys_prompt}\n"
                        f"\n"
                        f"Perfect diary generation prompt:\n{perfect_prompt_final}\n"
                        f"\n"
                        f"Inconsistency diary system prompt:\n{inconsistency_sys_prompt}\n"
                        f"\n"
                        f"Inconsistency diary generation prompt:\n{inconsistency_prompt_final}\n"
                        )
                print(f"Saved parameters to {OUTPUT_EXP_DIR}_{count}/{TRACK_FILE}_{patient_id}.txt")
                
            print("\n")
        else:
            print(f"Skipping patient {patient_id} as it has already been processed.")
        
        pbar.update(1)
    
        
pbar.close()

Generating synthetic clinical diaries:  17%|█▋        | 5/30 [06:48<34:00, 81.62s/it]


Processing patient: ./patient_profiles\patient_1.json
Skipping patient 1 as it has already been processed.
Processing patient: ./patient_profiles\patient_10.json
Skipping patient 10 as it has already been processed.
Processing patient: ./patient_profiles\patient_11.json
Skipping patient 11 as it has already been processed.
Processing patient: ./patient_profiles\patient_12.json
Skipping patient 12 as it has already been processed.
Processing patient: ./patient_profiles\patient_13.json
Skipping patient 13 as it has already been processed.
Processing patient: ./patient_profiles\patient_14.json
Generating diary for patient 14
Selected stylistic mode for patient 14: toxicity-focused
Selected length mode for patient 14: short
Saved LLM output on ./outputs/diary-gen_experiment_19/perfect-diary_patient_14.txt


Generating synthetic clinical diaries:  20%|██        | 6/30 [00:06<00:26,  1.09s/it]

Saved LLM output on ./outputs/diary-gen_experiment_19/inconsistancy-diary_patient_14.txt
Saved parameters to ./outputs/diary-gen_experiment_19/parameter_patient_14.txt


Processing patient: ./patient_profiles\patient_15.json
Generating diary for patient 15
Selected stylistic mode for patient 15: psychosocial-emphasis
Selected length mode for patient 15: short
Saved LLM output on ./outputs/diary-gen_experiment_19/perfect-diary_patient_15.txt


Generating synthetic clinical diaries:  23%|██▎       | 7/30 [00:56<04:02, 10.53s/it]

...

Generating synthetic clinical diaries:  53%|█████▎    | 16/30 [10:30<12:45, 54.67s/it]

Saved LLM output on ./outputs/diary-gen_experiment_19/inconsistancy-diary_patient_23.txt
Saved parameters to ./outputs/diary-gen_experiment_19/parameter_patient_23.txt


Processing patient: ./patient_profiles\patient_24.json
Generating diary for patient 24
Selected stylistic mode for patient 24: telegraphic-hospital-style
Selected length mode for patient 24: medium
Saved LLM output on ./outputs/diary-gen_experiment_19/perfect-diary_patient_24.txt


Generating synthetic clinical diaries:  57%|█████▋    | 17/30 [11:27<11:57, 55.23s/it]

Saved LLM output on ./outputs/diary-gen_experiment_19/inconsistancy-diary_patient_24.txt
Saved parameters to ./outputs/diary-gen_experiment_19/parameter_patient_24.txt


Processing patient: ./patient_profiles\patient_25.json
Generating diary for patient 25
Selected stylistic mode for patient 25: telegraphic-hospital-style
Selected length mode for patient 25: medium


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kbn0n3ddf6c8a9npft736hda` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 1415, Requested 6654. Please try again in 517.5ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
pbar = tqdm(total=len(PATIENT_PROFILES), desc="Generating synthetic Laboratory Analysis Reports")

for patient in PATIENT_PROFILES:
    print("Processing patient:", patient)
    with open(patient, "r", encoding="utf-8") as f, \
         open(ANALYSIS_PROMPT, "r", encoding="utf-8") as analysis_prompt_file, \
         open(ANALYSIS_SYS_PROMPT, "r", encoding="utf-8") as analysis_sys_prompt_file, \
         open(LAB_EXAMPLE, "r", encoding="utf-8") as lab_example_file, \
         open(f"{OUTPUT_EXP_DIR}_{count}/{PERFECT_OUTPUT_FILE}_{patient.split('_')[2].split('.')[0]}.txt", "r", encoding="utf-8") as perfect_diary_file, \
         open(LAB_SCHEMA, "r", encoding="utf-8") as lab_schema_file:
             
        patient_id = int(patient.split('_')[2].split('.')[0])
        if patient_id > 11:
            print(f"Generating analysis report for patient {patient_id}")

            patient_data = json.load(f)
        
            base_analysis_prompt = analysis_prompt_file.read()
            analysis_sys_prompt = analysis_sys_prompt_file.read()
            lab_example = lab_example_file.read()
            lab_schema = lab_schema_file.read()
            
            analysis_prompt_w_patient = base_analysis_prompt.replace("{{PATIENT_PROF}}", json.dumps(patient_data))
            analysis_prompt_w_clinical_diary = analysis_prompt_w_patient.replace("{{CLINICAL_DIARY}}", perfect_diary_file.read())
            analysis_prompt_w_lab_schema = analysis_prompt_w_clinical_diary.replace("{{LAB_SCHEMA}}", lab_schema)
            analysis_prompt_final = analysis_prompt_w_lab_schema.replace("{{ANALYSIS_REP}}", lab_example)
            
            
            completion = CLIENT.chat.completions.create(
                model=MODEL, # llama-3.3-70b-versatile, openai/gpt-oss-120b the prompt isn-t optimized for gpt-oss-120b
                messages=[
                    {
                        "role": "system",
                        "content": analysis_sys_prompt
                    },
                    {
                        "role": "user",
                        "content": analysis_prompt_final
                    }
                ],
                temperature=TEMP
            )
            analysis_result = completion.choices[0].message.content
            
            with open(f"{OUTPUT_EXP_DIR}_{count}/{ANALYSIS_OUTPUT_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
                o.write(f"{analysis_result}\n\n")
                print(f"Saved LLM output on {OUTPUT_EXP_DIR}_{count}/{ANALYSIS_OUTPUT_FILE}_{patient_id}.txt")
                
            output_path = f"{OUTPUT_EXP_DIR}_{count}/{TRACK_FILE}_{patient_id}.txt"

            with open(output_path, "a", encoding="utf-8") as o:
                o.write("\n")
                o.write(f"Analysis Report System Prompt:\n{analysis_result}\n")
                o.write("\n")
                o.write(f"Analysis Report Generation Prompt:\n{analysis_prompt_final}\n")
                o.write("\n\n")

                print(f"Saved Analysis report to {OUTPUT_EXP_DIR}_{count}/{TRACK_FILE}_{patient_id}.txt")
            
        else:
            print(f"Skipping patient {patient_id} as it has already been processed.")
        
        pbar.update(1)

Generating synthetic Laboratory Analysis Reports:   0%|          | 0/30 [00:00<?, ?it/s]

Processing patient: ./patient_profiles\patient_1.json
Skipping patient 1 as it has not been processed.
Processing patient: ./patient_profiles\patient_10.json
Skipping patient 10 as it has not been processed.
Processing patient: ./patient_profiles\patient_11.json


FileNotFoundError: [Errno 2] No such file or directory: './outputs/diary-gen_experiment_19/perfect-diary_patient_11.txt'